# ARTI 308 - Lab 5: Feature Engineering (Classification)
## Hit Status Prediction using the VG Sales Dataset

### Lab focus
This lab applies feature engineering on the `vgsales` dataset to build a classifier for whether a game is a **Hit** (`Global_Sales >= 1.0` million) or **Non-Hit**.

You will:
- inspect the dataset and target balance,
- engineer useful features,
- encode categorical variables,
- train and evaluate a baseline model,
- interpret feature importance.


## 1. Setup and imports


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Load the dataset


In [ ]:
DATA_PATH = "vgsales.csv"
df = pd.read_csv(DATA_PATH)

df.head(10)


The first rows confirm the dataset loaded correctly.  
Each row represents one video game title with metadata (platform, year, genre, publisher) and regional/global sales.


## 3. Quick dataset checks


In [ ]:
print("Shape:", df.shape)
print()
print("Missing values per column:")
display(df.isna().sum().to_frame("missing_count").T)

print()
print("Duplicate rows:", df.duplicated().sum())


The dataset has some missing values (notably in `Year` and `Publisher`).  
We will handle them through feature engineering and preprocessing instead of dropping large portions of data.


## 4. Target variable and class balance


In [ ]:
target_col = "Hit_Status"
df[target_col] = np.where(df["Global_Sales"] >= 1.0, "Hit", "Non-Hit")

df[target_col].value_counts()


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x=target_col, data=df)
plt.title("Hit Status Distribution")
plt.xlabel("Hit Status")
plt.ylabel("Count")
plt.show()


This chart shows class imbalance between `Hit` and `Non-Hit`.  
Because of imbalance, we will use stratified split and class-balanced modeling.


## 5. Identify feature types


In [ ]:
df.dtypes


We have mixed feature types:
- numerical: `Year`, sales columns, `Rank`
- categorical/text: `Name`, `Platform`, `Genre`, `Publisher`

This requires numerical scaling/imputation and categorical encoding.


## 6. Leakage awareness (important)


To predict `Hit_Status`, we should avoid leakage:
- `Global_Sales` directly defines the target and must not be used as an input feature.
- Regional sales (`NA_Sales`, `EU_Sales`, `JP_Sales`, `Other_Sales`) are strongly tied to global sales and are post-release outcomes.
- `Rank` is also driven by sales performance.

So we focus mainly on metadata and engineered features from metadata.


## 7. Feature engineering


### 7.1 Year-based features from `Year`
We create:
- `year_missing_flag`
- `Year_filled` (median-imputed year)
- `release_decade`
- `is_modern_release`


In [ ]:
df_fe = df.copy()

df_fe["year_missing_flag"] = df_fe["Year"].isna().astype(int)
year_median = df_fe["Year"].median()
df_fe["Year_filled"] = df_fe["Year"].fillna(year_median)
df_fe["release_decade"] = (df_fe["Year_filled"] // 10 * 10).astype(int).astype(str) + "s"
df_fe["is_modern_release"] = (df_fe["Year_filled"] >= 2005).astype(int)

df_fe[["Year", "Year_filled", "year_missing_flag", "release_decade", "is_modern_release"]].head(10)


These transformations convert a partially missing numeric field into several model-friendly features.


### 7.2 Title-based features from `Name`
We build lightweight text-derived features:
- `title_word_count`
- `title_has_number`
- `title_has_colon`


In [ ]:
df_fe["title_word_count"] = df_fe["Name"].str.split().str.len()
df_fe["title_has_number"] = df_fe["Name"].str.contains(r"\d", regex=True).astype(int)
df_fe["title_has_colon"] = df_fe["Name"].str.contains(":", regex=False).astype(int)

df_fe[["Name", "title_word_count", "title_has_number", "title_has_colon"]].head(10)


These simple NLP-style features can capture naming patterns (sequels, long franchise names, subtitle usage).


### 7.3 Interaction feature: `platform_genre`
A game's performance can depend on the platform-genre combination.  
We create a combined categorical feature.


In [ ]:
df_fe["platform_genre"] = df_fe["Platform"] + "_" + df_fe["Genre"]
df_fe[["Platform", "Genre", "platform_genre"]].head(10)


Interaction features often help tree models capture joint effects more directly.


### 7.4 Reducing high-cardinality categories (`Publisher`)
`Publisher` has many unique values. We keep top publishers and group the rest into `Other_Publisher`.


In [ ]:
if "Publisher" in df_fe.columns:
    top_k = 25
    df_fe["Publisher_filled"] = df_fe["Publisher"].fillna("Unknown")
    top_publishers = df_fe["Publisher_filled"].value_counts().head(top_k).index

    df_fe["Publisher_reduced"] = np.where(
        df_fe["Publisher_filled"].isin(top_publishers),
        df_fe["Publisher_filled"],
        "Other_Publisher"
    )

    display(df_fe["Publisher_reduced"].value_counts().head(10))


Reducing cardinality usually improves generalization and keeps one-hot dimensions manageable.


## 8. Discretization (binning)


We discretize release year into broad eras to capture coarse market generations.


In [ ]:
df_fe["release_period"] = pd.cut(
    df_fe["Year_filled"],
    bins=[1970, 1989, 1999, 2009, 2019, np.inf],
    labels=["classic", "90s", "2000s", "2010s", "2020+"],
    include_lowest=True
)

df_fe[["Year_filled", "release_period"]].head(10)


Binning can help represent non-linear effects and improve interpretability.


## 9. Prepare features for modeling


We drop leakage columns and raw high-cardinality text we already encoded through engineered features.


In [ ]:
drop_cols = [
    "Rank", "Name", "Publisher", "Publisher_filled",
    "NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales",
    "Year", "Hit_Status"
]
drop_cols = [c for c in drop_cols if c in df_fe.columns]

X = df_fe.drop(columns=drop_cols)
y = df_fe[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head(5)


The feature matrix now includes engineered metadata-driven predictors and excludes leakage-prone sales outcomes.


## 10. Split into train and test sets


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Stratified split preserves class proportions in train and test sets.


## 11. Encoding and baseline model (Random Forest)


### Why encoding and imputation?
- Models require numerical input, so categorical columns are one-hot encoded.
- Missing values are handled with `SimpleImputer`.
- We use `class_weight="balanced"` to account for class imbalance.


In [ ]:
categorical_cols = X_train.select_dtypes(include=["object", "category", "str"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)


## 12. Train the model and evaluate


In [ ]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred))


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=model.classes_,
    yticklabels=model.classes_
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


Accuracy gives a quick overall metric, while precision/recall/F1 show class-specific behavior, which is critical under imbalance.


## 13. Feature importance (What mattered the most?)


Random Forest provides feature importance scores.  
Because we one-hot encode categories, each category value gets its own derived feature.


In [ ]:
ohe = model.named_steps["preprocess"].named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = ohe.get_feature_names_out(categorical_cols) if len(categorical_cols) > 0 else np.array([])
all_feature_names = np.concatenate([numeric_cols, cat_feature_names])

importances = model.named_steps["classifier"].feature_importances_

fi = pd.DataFrame({
    "feature": all_feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

fi.head(20)


In [ ]:
plt.figure(figsize=(8, 5))
top_n = 15
sns.barplot(data=fi.head(top_n), x="importance", y="feature")
plt.title(f"Top {top_n} Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


Use this chart to interpret which engineered and original metadata features most influenced hit prediction.


## 14. Optional: Feature selection using SelectFromModel


We can optionally select stronger features using a model-based selector.


In [ ]:
from sklearn.feature_selection import SelectFromModel

selector = SelectFromModel(
    estimator=RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ),
    threshold="median"
)

model_fs = Pipeline(steps=[
    ("preprocess", preprocess),
    ("selector", selector),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

model_fs.fit(X_train, y_train)
y_pred_fs = model_fs.predict(X_test)

print("Accuracy with feature selection:", round(accuracy_score(y_test, y_pred_fs), 4))
print()
print("Classification report (feature selection model):")
print(classification_report(y_test, y_pred_fs))


If performance stays similar with fewer features, the model can become simpler and easier to maintain.


## 15. Student tasks


### Task 1 (Solved)
Additional engineered feature: `platform_age_at_release`

Definition:
`platform_age_at_release = Year_filled - first_release_year_of_platform`

Justification:
This feature approximates the lifecycle stage of each platform. Games released early in a platform's life can benefit from stronger launch momentum, while late-cycle releases may face smaller attention and demand. That lifecycle signal is not captured by `Year_filled` alone, so it can improve hit/non-hit separation.


In [ ]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression


def prepare_task_dataset(threshold=1.0, add_platform_age=False):
    d = df_fe.copy()
    d["Hit_Status_Task"] = np.where(df["Global_Sales"] >= threshold, "Hit", "Non-Hit")

    if add_platform_age:
        platform_first_year = d.groupby("Platform")["Year_filled"].transform("min")
        d["platform_age_at_release"] = (d["Year_filled"] - platform_first_year).clip(lower=0)

    drop_cols_task = [
        "Rank", "Name", "Publisher", "Publisher_filled",
        "NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales",
        "Year", "Hit_Status", "Hit_Status_Task"
    ]
    X_task = d.drop(columns=[c for c in drop_cols_task if c in d.columns])
    y_task = d["Hit_Status_Task"]

    return d, X_task, y_task


def evaluate_with_pipeline(X_task, y_task, estimator, random_state=42):
    X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
        X_task, y_task,
        test_size=0.2,
        random_state=random_state,
        stratify=y_task
    )

    cat_cols_t = X_train_t.select_dtypes(include=["object", "category", "str"]).columns.tolist()
    num_cols_t = X_train_t.select_dtypes(include=["number"]).columns.tolist()

    num_pipe_t = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    cat_pipe_t = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocess_t = ColumnTransformer(transformers=[
        ("num", num_pipe_t, num_cols_t),
        ("cat", cat_pipe_t, cat_cols_t)
    ])

    pipe_t = Pipeline(steps=[
        ("preprocess", preprocess_t),
        ("classifier", clone(estimator))
    ])

    pipe_t.fit(X_train_t, y_train_t)
    y_pred_t = pipe_t.predict(X_test_t)

    report_t = classification_report(y_test_t, y_pred_t, output_dict=True, zero_division=0)

    metrics_t = {
        "accuracy": accuracy_score(y_test_t, y_pred_t),
        "hit_precision": report_t["Hit"]["precision"],
        "hit_recall": report_t["Hit"]["recall"],
        "hit_f1": report_t["Hit"]["f1-score"],
        "macro_f1": report_t["macro avg"]["f1-score"],
    }

    return metrics_t, pipe_t


# Task 1 evaluation: baseline vs engineered feature
rf_task = RandomForestClassifier(
    n_estimators=250,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

df_task1_base, X_task1_base, y_task1_base = prepare_task_dataset(threshold=1.0, add_platform_age=False)
df_task1_feat, X_task1_feat, y_task1_feat = prepare_task_dataset(threshold=1.0, add_platform_age=True)

display(df_task1_feat[["Platform", "Year_filled", "platform_age_at_release"]].head(10))

m_base, _ = evaluate_with_pipeline(X_task1_base, y_task1_base, rf_task)
m_feat, _ = evaluate_with_pipeline(X_task1_feat, y_task1_feat, rf_task)

task1_results = pd.DataFrame([
    {"setup": "RF baseline (no platform_age_at_release)", **m_base},
    {"setup": "RF + platform_age_at_release", **m_feat}
]).set_index("setup")

display(task1_results.round(4))


Task 1 answer: the feature is implemented and evaluated. The comparison table above is the direct evidence for whether this engineered feature improves hit detection (`hit_recall`/`hit_f1`).


### Task 2 (Solved)
Changed threshold from `1.0` to `0.5` million, then compared class balance and model metrics.


In [ ]:
# Task 2: threshold sensitivity
_, X_t1, y_t1 = prepare_task_dataset(threshold=1.0, add_platform_age=True)
_, X_t05, y_t05 = prepare_task_dataset(threshold=0.5, add_platform_age=True)

balance_table = pd.DataFrame({
    "count_threshold_1.0": y_t1.value_counts(),
    "pct_threshold_1.0": y_t1.value_counts(normalize=True).round(4),
    "count_threshold_0.5": y_t05.value_counts(),
    "pct_threshold_0.5": y_t05.value_counts(normalize=True).round(4),
}).fillna(0)

display(balance_table)

m_t1, _ = evaluate_with_pipeline(X_t1, y_t1, rf_task)
m_t05, _ = evaluate_with_pipeline(X_t05, y_t05, rf_task)

task2_results = pd.DataFrame([
    {"threshold": "1.0 million", **m_t1},
    {"threshold": "0.5 million", **m_t05},
]).set_index("threshold")

display(task2_results.round(4))


Task 2 answer: lowering the threshold changes class balance and usually increases `Hit` support in training/testing, which affects precision-recall tradeoff.


### Task 3 (Solved)
Tried a second classifier (Logistic Regression) and compared it with Random Forest using the same engineered feature set.


In [ ]:
# Task 3: model comparison on the same dataset setup
_, X_task3, y_task3 = prepare_task_dataset(threshold=1.0, add_platform_age=True)

model_candidates = {
    "RandomForest": RandomForestClassifier(
        n_estimators=250,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ),
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="liblinear"
    )
}

task3_rows = []
for model_name, estimator in model_candidates.items():
    metrics_model, _ = evaluate_with_pipeline(X_task3, y_task3, estimator)
    task3_rows.append({"model": model_name, **metrics_model})

task3_results = pd.DataFrame(task3_rows).set_index("model").sort_values("hit_f1", ascending=False)
display(task3_results.round(4))

best_model = task3_results["hit_f1"].idxmax()
print(f"Best model by Hit F1: {best_model}")


Task 3 answer: the comparison table and `Best model by Hit F1` line provide the final classifier choice under this task's evaluation criterion.


## Wrap-up
In this lab, we adapted feature engineering workflow to the `vgsales` dataset:
- built year-, title-, interaction-, and cardinality-reduction features,
- avoided leakage from sales-driven columns,
- trained and evaluated a baseline classifier,
- interpreted model behavior via feature importance.
